# Fine-tune FunctionGemma for your Home Mind

This trains Google's FunctionGemma (270M parameters) to turn requests like *"dim the den lamp to 40"* into one of Home Mind's control tools, using **your house's own device names**. The result is small enough to run on an old laptop's CPU.

Based on Google's [FunctionGemma fine-tuning guide](https://ai.google.dev/gemma/docs/functiongemma/finetuning-with-functiongemma) (TRL `SFTTrainer`, full fine-tune).

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (free tier is enough).
2. Accept the Gemma terms at https://huggingface.co/google/functiongemma-270m-it and make a Hugging Face read token.
3. Download your training data from Home Mind (admin token): `GET /api/control_model/dataset?part=train` and `?part=test` (add `&paraphrase=1` to the train link for more varied wording from your general model). You get `home_mind_control_train.jsonl` and `home_mind_control_test.jsonl`. They contain your device names, nothing else from your memory.

Training takes about 10–30 minutes on a T4 depending on how many devices you have.

In [ ]:
%pip install -q "torch" "transformers>=4.57" "datasets" "accelerate" "trl>=0.24" "sentencepiece" "protobuf"


In [ ]:
from huggingface_hub import login
login()  # paste your Hugging Face read token (Gemma terms must be accepted)


## 1. Upload your data

In [ ]:
from google.colab import files
up = files.upload()  # pick home_mind_control_train.jsonl and home_mind_control_test.jsonl
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "home_mind_control_train.jsonl", "test": "home_mind_control_test.jsonl"})
print(ds)
print(ds["train"][0]["messages"])


## 2. Load FunctionGemma and check the prompt format

In [ ]:
import torch, json, re
from transformers import AutoModelForCausalLM, AutoTokenizer
base_model = "google/functiongemma-270m-it"
model = AutoModelForCausalLM.from_pretrained(base_model, dtype="auto", device_map="auto", attn_implementation="eager")
tokenizer = AutoTokenizer.from_pretrained(base_model)
ex = ds["train"][0]
print(tokenizer.apply_chat_template(ex["messages"], tools=ex["tools"], tokenize=False)[-600:])


## 3. Score before training

Exact match on the held-out test set: right tool and right arguments (device words, numbers, colors). The test set uses wordings and some devices that never appear in training.

In [ ]:
CALL = re.compile(r"call:(\w+)\{(.*?)\}", re.S)
ARG = re.compile(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]*))", re.S)

def predict(messages, tools):
    inputs = tokenizer.apply_chat_template(messages, tools=tools, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=False)
    m = CALL.search(text)
    if not m:
        return None, {}, text
    args = {}
    for a in ARG.finditer(m.group(2)):
        v = (a.group(2) if a.group(2) is not None else a.group(3) or "").strip()
        args[a.group(1)] = int(v) if v.isdigit() else v
    return m.group(1), args, text

def words(s):
    return set(w for w in re.findall(r"[a-z0-9]+", str(s).lower()) if w not in ("the", "a", "my", "our", "in", "of"))

def evaluate(rows, show=5):
    ok, bad = 0, []
    for r in rows:
        want = r["messages"][-1]["tool_calls"][0]["function"]
        tool, args, text = predict(r["messages"][:-1], r["tools"])
        good = tool == want["name"]
        if good and tool != "not_a_command":
            for k, v in want["arguments"].items():
                good &= (words(args.get(k)) == words(v)) if k == "device" else str(args.get(k)).lower() == str(v).lower()
        ok += good
        if not good and len(bad) < show:
            bad.append((r["messages"][-2]["content"], want, text[:120]))
    for b in bad:
        print("MISS", b)
    return round(ok / len(rows), 3)

print("before:", evaluate(ds["test"]))


## 4. Train

In [ ]:
from trl import SFTConfig, SFTTrainer
args = SFTConfig(
    output_dir="functiongemma-home-mind",
    max_length=2048,            # the tool list is long: 13 tools
    packing=False,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    lr_scheduler_type="constant",
    optim="adamw_torch_fused",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
                     processing_class=tokenizer)
trainer.train()
model = trainer.model.eval()
print("after:", evaluate(ds["test"]))


If *after* is below about 0.9, add `&paraphrase=1` to the train download, or raise `num_train_epochs` to 5, and train again.

## 5. Save and convert for llama.cpp / Ollama (GGUF)

In [ ]:
trainer.save_model("functiongemma-home-mind")
tokenizer.save_pretrained("functiongemma-home-mind")
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
%pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py functiongemma-home-mind --outtype q8_0 --outfile home-mind-control.gguf
!ls -lh home-mind-control.gguf
from google.colab import files
files.download("home-mind-control.gguf")


## 6. Run it at home

The file is about 300 MB. On the laptop (or any machine Home Mind can reach), run llama.cpp's server, which uses the model's own chat template:

```
llama-server -m home-mind-control.gguf --jinja --port 8081 --host 0.0.0.0 -c 4096
```

Then in Home Mind's options add a model and point `control_model` at it:

```yaml
models:
  - name: control
    type: openai
    url: http://LAPTOP-IP:8081/v1
control_model: control
```

Home Mind sends each house request to it first. It only *proposes* a tool call: the device name is matched against your control list, locks and garage doors still need your "yes", and anything it marks `not_a_command` (questions, several actions at once) goes to your normal model. Retrain whenever you add or rename devices.